In [2]:
import json
import os
import shutil
from pathlib import Path

import yaml
from pyoxigraph import Store, RdfFormat, Dataset, parse
from pyshacl import validate


SRCDIR = Path(".")
BUILDDIR = SRCDIR / ".." / "build"
DATADIR = SRCDIR / ".." / "data"

print(os.path.abspath(DATADIR))


def yaml2json(y):
    os.makedirs(BUILDDIR / y.parent, exist_ok=True)
    with y.open("rb") as f:
        data = yaml.safe_load(f)
    with open(BUILDDIR / y.with_suffix(".jsonld"), "w") as g:
        json.dump(data, g, default=str)


shutil.rmtree(BUILDDIR, ignore_errors=True)

yaml2json(SRCDIR / "remote" / "base-voltage.shacl.yamlld")
yaml2json(SRCDIR / "remote" / "nbnl-prof.context.yamlld")
yaml2json(SRCDIR / "nbnl-equipment.rdfs.yamlld")
yaml2json(SRCDIR / "nbnl-equipment.prof.yamlld")
shutil.copy(SRCDIR / "remote" / "cim17.rdfs.ttl", BUILDDIR / "remote")
shutil.copy(SRCDIR / "remote" / "nbnl-prof.owl.ttl", BUILDDIR / "remote")

data_dataset = Dataset()
with open(
    DATADIR / "AS_EQ_OP_Westland_FGU" / "equipment_four_stations_EQ.jsonld", "rb"
) as f:
    data_dataset.bulk_load(f.read, format=RdfFormat.JSON_LD)

shapes_dataset = Dataset()
with open(BUILDDIR / "nbnl-equipment.shacl.jsonld", "rb") as f:
    shapes_dataset.bulk_load(f.read(), format=RdfFormat.JSON_LD)

ontology_graph = Dataset()
with open(BUILDDIR / "nbnl-equipment.rdfs.jsonld", "rb") as f:
    ontology_graph.bulk_load(f.read(), format=RdfFormat.JSON_LD)

conforms, results_graph, results_text = validate(
    data_graph=data_dataset,
    shacl_graph=shapes_graph,
    ont_graph=ontology_graph,  # TODO: This is very specific to this validator. Can we do this differently?
    meta_shacl=True,
    inference="rdfs",
    do_owl_imports=True,
)

print(results_text)

/home/bart/Programming/netbeheernederland/poc-nbnl-equipment/data


AttributeError: 'pyoxigraph.Dataset' object has no attribute 'bulk_load'